In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(model=model, lr_init=1e-3)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5615185499191284
epoch:  1, loss: 0.5588390231132507
epoch:  2, loss: 0.5561724305152893
epoch:  3, loss: 0.5535183548927307
epoch:  4, loss: 0.5508766174316406
epoch:  5, loss: 0.5482470989227295
epoch:  6, loss: 0.5456298589706421
epoch:  7, loss: 0.5430248379707336
epoch:  8, loss: 0.5404319763183594
epoch:  9, loss: 0.5378511548042297
epoch:  10, loss: 0.5352822542190552
epoch:  11, loss: 0.5327252149581909
epoch:  12, loss: 0.5301802158355713
epoch:  13, loss: 0.5276471972465515
epoch:  14, loss: 0.5251259803771973
epoch:  15, loss: 0.522616446018219
epoch:  16, loss: 0.520118772983551
epoch:  17, loss: 0.5176327228546143
epoch:  18, loss: 0.5151582956314087
epoch:  19, loss: 0.5126954913139343
epoch:  20, loss: 0.5102440714836121
epoch:  21, loss: 0.5078039169311523
epoch:  22, loss: 0.505375325679779
epoch:  23, loss: 0.5029580593109131
epoch:  24, loss: 0.5005519986152649
epoch:  25, loss: 0.4981573224067688
epoch:  26, loss: 0.4957738220691681
epoch:  27, lo

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -3563.4399652731227
Test metrics:  R2 = -3533.2445894904604


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradientLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.11795807629823685
epoch:  1, loss: 0.07801838964223862
epoch:  2, loss: 0.057351164519786835
epoch:  3, loss: 0.04634188488125801
epoch:  4, loss: 0.04041853919625282
epoch:  5, loss: 0.03722932189702988
epoch:  6, loss: 0.03551381453871727
epoch:  7, loss: 0.034591514617204666
epoch:  8, loss: 0.034095343202352524
epoch:  9, loss: 0.033827222883701324
epoch:  10, loss: 0.033681172877550125
epoch:  11, loss: 0.03360017389059067
epoch:  12, loss: 0.03355375677347183
epoch:  13, loss: 0.033525411039590836
epoch:  14, loss: 0.03349003195762634
epoch:  15, loss: 0.03344959393143654
epoch:  16, loss: 0.033425021916627884
epoch:  17, loss: 0.03341483324766159
epoch:  18, loss: 0.03337946906685829
epoch:  19, loss: 0.03335949778556824
epoch:  20, loss: 0.033347420394420624
epoch:  21, loss: 0.033332157880067825
epoch:  22, loss: 0.033314671367406845
epoch:  23, loss: 0.03331417962908745
epoch:  24, loss: 0.03328646346926689
epoch:  25, loss: 0.033270444720983505
epoch:  26,

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6634225031237247
Test metrics:  R2 = 0.6705945373869756
